# 第7回：数値を予測する—回帰

**今日の問い：連続値の予測モデルを、何と比べ、不確かさをどう示すか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。`DEEP DIVE`は経験者や自習向けの発展です。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- MAE・RMSE・R²を異なる視点として読み、複数モデルを同条件で比較する
- 学習曲線と残差診断から、データ不足か表現力不足かを切り分ける
- 分位点回帰やブートストラップで予測の不確かさを区間として示す

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- MAE：絶対誤差の平均
- RMSE：大きな誤差をより重く扱う指標
- 残差：実測値と予測値の差
- 学習曲線：データ量に対する性能の変化
- 予測区間：予測値に付ける不確かさの幅

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


## 回帰＝「数値そのもの」を予測する

ここまでは活性の有無（0/1）でしたが、この回は**収率（%）という連続値**を予測します。これを
**回帰**と呼びます。回帰でいちばん大事な問いは「その予測は**何と比べて**良いのか」。だから今回も
**平均値だけを返すベースライン**を必ず土俵に上げます。

まず下準備。日本語フォント設定と、使うモデルの読み込み、学習/検証の分割をまとめて行います。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
for _name in ["Yu Gothic", "Meiryo", "Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]:
    if _name in {f.name for f in font_manager.fontManager.ttflist}:
        plt.rcParams["font.family"] = _name
        break
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

features = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
X_train, X_valid, y_train, y_valid = train_test_split(df[features], df["yield_pct"], test_size=0.25, random_state=42)


## TRY：4モデルを交差検証で、3つの指標で比べる

回帰の代表的な指標を先に押さえます。

- **MAE（平均絶対誤差）**：平均で何%外すか。単位が収率と同じで**いちばん直感的**。
- **RMSE**：大きな外れを二乗で重く見る。「たまの大外し」を嫌う場面向け。
- **R²（決定係数）**：平均値予測と比べてどれだけ説明できたか（1に近いほど良い、0は平均値並み）。

`neg_...`はsklearnの都合で「大きいほど良い」に符号反転された指標名。表示時に`-`で元へ戻します。


In [ ]:
from sklearn.model_selection import cross_validate, KFold

models = {
    "平均値": DummyRegressor(),
    "線形回帰": make_pipeline(SimpleImputer(strategy="median"), LinearRegression()),
    "決定木": make_pipeline(SimpleImputer(strategy="median"), DecisionTreeRegressor(max_depth=4, random_state=42)),
    "Random Forest": make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)),
}
cv = KFold(5, shuffle=True, random_state=42)
scoring = {"MAE": "neg_mean_absolute_error", "RMSE": "neg_root_mean_squared_error", "R2": "r2"}
rows = []
for name, estimator in models.items():
    res = cross_validate(estimator, df[features], df["yield_pct"], cv=cv, scoring=scoring)
    rows.append({"モデル": name, "MAE": -res["test_MAE"].mean(), "RMSE": -res["test_RMSE"].mean(), "R2": res["test_R2"].mean()})
pd.DataFrame(rows).sort_values("MAE").round(3)


### 出力の読み方

- MAEの小さい順に並びます。**「平均値」より各モデルがどれだけMAEを下げたか**が価値です。下げ幅が小さいなら、その特徴量では収率を説明しきれていません。
- MAEとRMSEの差が大きいモデルは、**たまに大きく外している**サイン（RMSEが大外れを強調するため）。
- R²が0近くなら「平均値と大差ない」、負なら「平均値より悪い」。**まずベースライン超え**を確認します。


## 予測と実測、そして「残差」を絵で見る

数字だけでなく図で確かめます。左は**予測と実測の散布図**（点が対角線に乗るほど良い）、右は
**残差図**（実測−予測を予測値に対してプロット）。残差は0の周りに**模様なくばらける**のが理想です。
偏りや傾きがあれば、モデルが取りこぼした構造があります。


In [ ]:
rf = make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)).fit(X_train, y_train)
pred = rf.predict(X_valid)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(y_valid, pred, alpha=0.6)
axes[0].plot([y_valid.min(), y_valid.max()], [y_valid.min(), y_valid.max()], "--")
axes[0].set(xlabel="実測収率", ylabel="予測収率", title="予測と実測")
axes[1].scatter(pred, y_valid - pred, alpha=0.6)
axes[1].axhline(0, linestyle="--")
axes[1].set(xlabel="予測収率", ylabel="残差（実測-予測）", title="残差")
plt.tight_layout()


### 出力の読み方

- **左**：点が破線（予測＝実測）に近いほど良い。高収率の領域で点が下に外れていれば、**高い収率を低めに予測しがち**という癖です。
- **右**：残差が予測値によって偏る（例：予測が大きいほど残差が下がる）なら、モデルが端の領域を苦手にしています。**きれいな水平の帯**が理想。
- 図は「どこで外すか」を教えてくれます。次のセルで、実際に大きく外した試料を取り出します。


## 大きく外した試料を名指しで調べる

平均のMAEでは見えない「個別の大外し」を確認します。絶対誤差の大きい上位8件を、系列や触媒つきで
取り出すと、**特定の条件で外していないか**の手がかりになります。


In [ ]:
errors = df.loc[y_valid.index, ["sample_id", "scaffold_group", "catalyst", "yield_pct"]].copy()
errors["予測"] = pred
errors["絶対誤差"] = (errors["yield_pct"] - errors["予測"]).abs()
errors.nlargest(8, "絶対誤差").round(2)


### 出力の読み方

大外し8件に**同じ系列や同じ触媒が偏っていないか**を見ます。偏っていれば、その条件を表す特徴量が
足りない可能性（第11回の特徴量設計の動機）。ばらばらなら、単なるノイズかもしれません。

## CHANGE

`max_depth=6`を`3`や`10`へ変え、MAEの表・残差図・大外し試料が**どう連動して動くか**を観察します。


## DEEP DIVE：伸び悩みの原因と、予測の不確かさ

発展として3つ。**学習曲線**（データを増やせば改善するか）、**群別残差**（どの系列で系統的に外すか）、
**予測区間**（1点の予測に幅を添える）です。


### 学習曲線：データ不足か、表現力不足か

「もっとデータを集めれば精度が上がる？」に答える図です。学習データ量を増やしながら、学習MAEと
検証MAEの推移を描きます。2本が近づいて高止まりなら**データ追加は効きにくい**（特徴量やモデルを
見直すべき）。2本が離れて検証MAEがまだ下がりそうなら**データ追加が効く**サインです。


In [ ]:
import numpy as np
from sklearn.model_selection import learning_curve

sizes, train_scores, valid_scores = learning_curve(
    make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)),
    df[features], df["yield_pct"], cv=5, scoring="neg_mean_absolute_error",
    train_sizes=np.linspace(0.2, 1.0, 5),
)
plt.figure(figsize=(7, 4))
plt.plot(sizes, -train_scores.mean(1), "o-", label="学習MAE")
plt.plot(sizes, -valid_scores.mean(1), "o-", label="検証MAE")
plt.xlabel("学習データ件数"); plt.ylabel("MAE"); plt.legend(); plt.title("学習曲線")
plt.tight_layout()
print("2本が高止まりで近いなら、データ追加より特徴量やモデルを見直す。")


### 出力の読み方

- 右端（全データ使用）で**検証MAEがまだ下降中**なら、データを増やす価値あり。**水平に寝ている**なら頭打ち。
- 学習MAEと検証MAEの**縦の隙間**が過学習の程度。隙間が大きいほど「覚えすぎ」寄りです。


### 群別に残差を見る：どの系列で偏るか

全体のMAEが良くても、特定の化合物系列だけ系統的に外していることがあります。系列ごとに件数・MAE・
**平均残差**（符号つき）を出すと、「この系列を平均的に低く見積もっている」といった偏りが見えます。


In [ ]:
errors["残差"] = errors["yield_pct"] - errors["予測"]
group_error = errors.groupby("scaffold_group").agg(件数=("残差", "size"), MAE=("絶対誤差", "mean"), 平均残差=("残差", "mean"))
display(group_error.sort_values("MAE", ascending=False).round(2))
print("平均残差が正なら、その系列を平均的に過小予測している。")


### 出力の読み方

**平均残差が0から大きく離れた系列**が要注意。正なら過小予測、負なら過大予測です。件数が少ない系列は
偶然も大きいので、件数と併せて読みます。系統的な偏りは、その系列を表す特徴量の不足を示唆します。


### 予測区間：1点でなく「幅」で答える

「収率は62%」より「10〜90%の確率で50〜74%」の方が、意思決定に誠実なことがあります。**分位点回帰**で
下限（10%点）と上限（90%点）を別々に予測し、実測が本当にその区間に入る割合（**被覆率**）を検証します。


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

low = HistGradientBoostingRegressor(loss="quantile", quantile=0.1, max_iter=200, random_state=42).fit(X_train, y_train)
high = HistGradientBoostingRegressor(loss="quantile", quantile=0.9, max_iter=200, random_state=42).fit(X_train, y_train)
low_pred, high_pred = low.predict(X_valid), high.predict(X_valid)
coverage = ((y_valid.to_numpy() >= low_pred) & (y_valid.to_numpy() <= high_pred)).mean()
print(f"10-90%予測区間の実測被覆率: {coverage:.1%}（理想は約80%）")
pd.DataFrame({"実測": y_valid.to_numpy()[:8], "下限": low_pred[:8].round(1), "上限": high_pred[:8].round(1)})


### 出力の読み方

- 10〜90%区間なので、被覆率は**理想80%**に近いほど区間が正直。大きく下回れば区間が狭すぎ（自信過剰）、上回れば広すぎです。
- 表の8件で、**実測が下限〜上限に収まっているか**を目で確認します。幅の広い試料はモデルが自信を持てていない試料です。


## よくある誤り

- R²だけで利用可能と判断する
- テストデータでモデルを選ぶ
- 点予測だけを示し不確かさを伝えない

## SELF-STUDY（任意・30〜60分）

- 学習曲線を描き、データ追加が効くかを1文で判断する
- 分位点回帰の10-90%区間の被覆率を検証データで確認する

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. MAEとRMSEは何を違って重視するか
2. 学習曲線から何を読み取れるか
3. 予測区間が点予測より役立つ場面はどこか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
